# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Kunal Ranjan  
**`Roll Number`:**  U20230022
**`GitHub Branch`:** Kunal_U20230022

# Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler


# Load Datasets

In [2]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [4]:
print("Missing values in train_users:\n", train_users.isnull().sum())
print("\nMissing values in test_users:\n", test_users.isnull().sum())
#No missing values

#Encoding
le = LabelEncoder()
train_users['label_encoded'] = le.fit_transform(train_users['label'])
test_users['label_encoded'] = le.transform(test_users['label'])

print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


feature_cols = ['age', 'income', 'clicks', 'purchase_amount']

X_train = train_users[feature_cols].values
y_train = train_users['label_encoded'].values

X_test = test_users[feature_cols].values
y_test = test_users['label_encoded'].values

print(f"\nTraining set: X={X_train.shape}, y={y_train.shape}")
print(f"Test set:     X={X_test.shape}, y={y_test.shape}")
print(f"\nClass distribution (train):\n{train_users['label'].value_counts()}")

Missing values in train_users:
 user_id            0
age                0
income             0
clicks             0
purchase_amount    0
label              0
label_encoded      0
dtype: int64

Missing values in test_users:
 user_id            0
age                0
income             0
clicks             0
purchase_amount    0
label              0
label_encoded      0
dtype: int64

Label mapping: {'user1': np.int64(0), 'user2': np.int64(1), 'user3': np.int64(2)}

Training set: X=(2000, 4), y=(2000,)
Test set:     X=(2000, 4), y=(2000,)

Class distribution (train):
label
user1    687
user2    669
user3    644
Name: count, dtype: int64


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
#Feature Engineering 
train_users['income_per_click'] = train_users['income'] / (train_users['clicks'] + 1)
train_users['purchase_per_click'] = train_users['purchase_amount'] / (train_users['clicks'] + 1)
train_users['age_income_ratio'] = train_users['age'] / (train_users['income'] + 1)

test_users['income_per_click'] = test_users['income'] / (test_users['clicks'] + 1)
test_users['purchase_per_click'] = test_users['purchase_amount'] / (test_users['clicks'] + 1)
test_users['age_income_ratio'] = test_users['age'] / (test_users['income'] + 1)

feature_cols_v2 = feature_cols + ['income_per_click', 'purchase_per_click', 'age_income_ratio']

X_train = train_users[feature_cols_v2].values
X_test = test_users[feature_cols_v2].values
#Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Decision Tree
dt_clf = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_clf.fit(X_train_scaled, y_train)
dt_preds = dt_clf.predict(X_test_scaled)
dt_acc = accuracy_score(y_test, dt_preds)

#Logistic Regr
lr_clf = LogisticRegression(max_iter=1000, random_state=42)
lr_clf.fit(X_train_scaled, y_train)
lr_preds = lr_clf.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_preds)

#Random Forest
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
rf_clf.fit(X_train_scaled, y_train)
rf_preds = rf_clf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_preds)

#Comparison
print("Model Comparison")
print(f"Decision Tree  accuracy: {dt_acc:.4f}")
print(f"Logistic Reg.  accuracy: {lr_acc:.4f}")
print(f"Random Forest  accuracy: {rf_acc:.4f}")

#Best pick
best_name, best_acc, best_clf, uses_scaler = max(
    [("Decision Tree", dt_acc, dt_clf, False),
     ("Logistic Regression", lr_acc, lr_clf, True),
     ("Random Forest", rf_acc, rf_clf, False)],
    key=lambda x: x[1]
)

#Storing the best classifier for later use
context_classifier = best_clf

def predict_user_context(features):
    """Predict user category (0=user1, 1=user2, 2=user3) from raw features."""
    features = np.array(features).reshape(1, -1)
    if uses_scaler:
        features = scaler.transform(features)
    return context_classifier.predict(features)[0]

'''Data is noisy. The feature distributions for User1,2,3 are all heavily overlapping so the model
accuracy is similar to guessing.'''

Model Comparison
Decision Tree  accuracy: 0.3340
Logistic Reg.  accuracy: 0.3205
Random Forest  accuracy: 0.3120


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
